# Semana 13 — Começando pela validação de mensagens no Node-RED

Como parte da turma ainda não concluiu o exercício anterior, vamos partir de um fluxo mínimo e executável: dois nós **Inject** publicam um payload válido e um inválido; um **Function** confere os campos obrigatórios; dois **Debug** mostram os resultados. Depois, ampliaremos está base para integrar a validação ao fluxo industrial da aula anterior e discutir alertas com estado e ciclo de vida.

## 1. Montagem do fluxo-base

Crie ou importe uma aba com estes nós e ligações:

```text
Inject -- mensagem completa --+--> Function: Validar telemetria --+--> Debug: Valida
Inject -- temperatura ausente -+                              +--> Debug: Invalida
```

Os dois Injects devem chegar à entrada da mesma Function. Configure a Function com **2 saídas**. A saída 1 representa mensagem aceita; a saída 2 representa mensagem rejeitada.


### Configuração do Inject 1 — Publicador Manual válido

No editor, abra o Inject e configure:

- **Name:** `Publicador Manual - válido`
- **msg.payload:** tipo **JSON**, valor abaixo
- **msg.topic:** tipo **string**, valor `fábrica/linha1/prensa_teste/telemetria`
- **Repeat:** desativado; **Inject once after start:** desativado

```json
{"máquina":"PRENSA_TESTE","temperatura":72,"vibração":3.2,"pressão":6.1}
```

Escolher o tipo JSON faz `msg.payload` chegar como objeto, permitindo acessar `msg.payload.máquina` e os demais campos na Function.


### Configuração do Inject 2 — payload sem temperatura

Duplique o primeiro Inject e altere:

- **Name:** `Publicador Manual - temperatura ausente`
- **msg.payload:** tipo **JSON**, valor abaixo
- **msg.topic:** mantenha `fábrica/linha1/prensa_teste/telemetria`
- **Repeat:** desativado; **Inject once after start:** desativado

```json
{"máquina":"PRENSA_TESTE","vibração":3.2,"pressão":6.1}
```

Os dois Injects usam o mesmo tópico para representar duas leituras da mesma rota. Ao clicar no botão de cada nó, a mensagem correspondente segue pelo fluxo.


## 2. Contrato da telemetria e validação

O payload deve conter os quatro campos abaixo. Configure o Function com **2 saídas** e use o código fornecido: a primeira saída recebe mensagens aceitas; a segunda, mensagens com campos ausentes. O código verifica presença dos campos.


In [ ]:
const p = msg.payload || {};
const obrigatorios = ["maquina", "temperatura", "vibracao", "pressao"];
const ausentes = obrigatorios.filter(campo => p[campo] === undefined);

if (ausentes.length > 0) {
    msg.validacao = {
        valido: false,
        motivo: "campos obrigatorios ausentes",
        campos: ausentes
    };
    return [null, msg]; // saida 2: invalida
}

msg.validacao = { valido: true, motivo: "contrato aceito" };
return [msg, null]; // saida 1: valida

### Debugs e resultado esperado

Ligue a **saída 1** a um Debug chamado `Debug - valida` e a **saída 2** a `Debug - inválida`. Em cada Debug, selecione a propriedade `msg.validacao` (ou a mensagem completa se quiser inspecionar todo o objeto).

Ao clicar no Inject válido, o Debug da saída 1 deve mostrar `válido: true` e `motivo: "contrato aceito"`. Ao clicar no segundo Inject, o Debug da saída 2 deve mostrar `válido: false` e `campos: ["temperatura"]`. O caminho de saída 2 é a mensagem de erro controlado; não é uma falha do Node-RED.

> Importante: na Function, `return [msg, null]` envia para a primeira saída; `return [null, msg]` envia para a segunda. A ordem dos fios no editor deve corresponder à ordem das saídas.

## 2. Evolução: integrar ao fluxo de telemetria

O fluxo `fluxo Node-RED de telemetria` já recebe `fábrica/#`, converte JSON, roteia por `msg.topic` (`telemetria`, `alarmes`, `producao`) e exibe Debugs. Agora use a validação acima no ramo `telemetria`, antes da regra de negócio. O Inject do fluxo-base simula a publicação MQTT localmente; ele não passa pelo broker, mas permite aprender a estrutura `msg` e depurar a Function.


## Objetivos

- Explicar QoS 1, duplicidade e `event_id`.
- Montar o fluxo de validação com Inject, Function e Debug.
- Validar sem transformar dados inválidos em zero.
- Implementar persistência, histerese, severidade e supressão de notificações.
- Distinguir alerta aberto, reconhecido e normalizado e testar transições.


## 3. Persistência, histerese e ciclo de vida

- **Persistência:** três amostras altas consecutivas; antes de abrir, uma amostra abaixo de 75 °C reinicia a contagem.
- **Histerese:** abre em ≥75 °C e normaliza em ≤70 °C; entre os limites mantém o estado atual.
- **Supressão:** leituras enquanto o alerta está ativo não criam novas aberturas.
- **Recorrência:** após normalização, nova persistência cria uma nova ocorrência.

Reconhecer é registrar que alguém viu o alerta; não altera a condição física. Estado em memória é didático e pode se perder em reinício. Para múltiplas instâncias, use armazenamento compartilhado e atualizações idempotentes.


In [ ]:
LIMITE_ABERTURA, LIMITE_NORMALIZACAO, ALTAS_PARA_ABRIR = 75.0, 70.0, 3
estado = {"status":"NORMAL", "altas":0, "alerta_id":None, "severidade":None}

def processar(estado, equipamento, event_id, temperatura):
    eventos = []
    if estado["status"] == "NORMAL":
        estado["altas"] = estado["altas"] + 1 if temperatura >= LIMITE_ABERTURA else 0
        if estado["altas"] >= ALTAS_PARA_ABRIR:
            estado.update(status="ABERTO", alerta_id=f"ALT-{equipamento}-{event_id}",
                          severidade="CRÍTICA" if temperatura >= 90 else "ALTA")
            eventos.append({"tipo":"ALERTA_ABERTO", "alerta_id":estado["alerta_id"],
                            "severidade":estado["severidade"]})
    elif temperatura <= LIMITE_NORMALIZACAO:
        eventos.append({"tipo":"ALERTA_NORMALIZADO", "alerta_id":estado["alerta_id"]})
        estado.update(status="NORMAL", altas=0, alerta_id=None, severidade=None)
    elif temperatura >= 90 and estado["severidade"] != "CRÍTICA":
        estado["severidade"] = "CRÍTICA"
        eventos.append({"tipo":"SEVERIDADE_ESCALADA", "alerta_id":estado["alerta_id"]})
    return eventos

for n, temp in enumerate([76.1, 77, 78.4, 74.9, 72, 69.5, 76, 77, 78], 1):
    saída = processar(estado, "prensa-01", f"evt-{n:03}", temp)
    print(n, temp, estado["status"], saída or "sem transicao")


Na terceira alta o alerta abre. 74,9 °C e 72 °C não o fecham; 69,5 °C normaliza. A sequência final abre uma nova ocorrência. A regra didática de 90 °C escala a severidade mesmo com alerta já aberto.

## 4. Deduplicação de eventos

QoS 1 significa entrega ao menos uma vez: uma publicação pode ser retransmitida. Deduplicar pelo `event_id` **antes** de atualizar a contagem impede que uma repetição seja contada como duas amostras.


In [ ]:
from collections import deque

class Deduplicador:
    def __init__(self, capacidade=1000):
        self.ids, self.ordem, self.capacidade = set(), deque(), capacidade
    def primeira_vez(self, event_id):
        if event_id in self.ids:
            return False
        self.ids.add(event_id); self.ordem.append(event_id)
        if len(self.ordem) > self.capacidade:
            self.ids.remove(self.ordem.popleft())
        return True

dedup = Deduplicador()
for eid in ["evt-1", "evt-2", "evt-2", "evt-3"]:
    print(eid, "processar" if dedup.primeira_vez(eid) else "duplicata ignorada")


A janela limitada evita crescimento ilimitado, mas IDs antigos podem ser aceitos após expirar. Em operação, retenha IDs por tempo superior à janela de retransmissão/reprocessamento e persista o registro conforme os requisitos.

## 5. Reconhecimento

Registre operador e horário. O reconhecimento não fecha o alerta e não apaga evidências.


In [ ]:
from datetime import datetime, timezone
alerta = {"status":"ABERTO", "alerta_id":"ALT-prensa-01-evt-003", "reconhecido_por":None, "reconhecido_em":None}
def reconhecer(alerta, usuario):
    if alerta["status"] != "ABERTO":
        raise ValueError("nao ha alerta ativo para reconhecer")
    if alerta["reconhecido_por"] is None:
        alerta["reconhecido_por"] = usuario
        alerta["reconhecido_em"] = datetime.now(timezone.utc).isoformat()
    return alerta
reconhecer(alerta, "operador.turno_a")


## 6. Function node para o ramo de telemetria

Substitua ou complemente o Function **Tratamento de Telemetria** depois do Switch. Configure-o com **2 saídas**: saída 1 para transições de alerta; saída 2 para telemetria válida sem transição. Conecte Debug nodes distintos. `node.error` permite tratar inválidas com um Catch node. O exemplo usa contexto local volátil, apropriado para laboratório.


In [ ]:
// Node-RED Function (2 saídas): [evento de alerta, leitura sem transicao]
const p = msg.payload;
if (!p || typeof p !== "object" || Array.isArray(p)) {
    node.error("payload deve ser objeto JSON", msg); return null;
}
const id = p.event_id, equipamento = p.equipment_id || p.maquina;
const temp = p.temperature_c ?? p.temperatura;
if (typeof id !== "string" || !id.trim() || typeof equipamento !== "string" ||
    !equipamento.trim() || typeof temp !== "number" || !Number.isFinite(temp)) {
    node.error("event_id, equipamento e temperatura numérica sao obrigatorios", msg);
    return null;
}
if (p.timestamp && Number.isNaN(Date.parse(p.timestamp))) {
    node.error("timestamp inválido", msg); return null;
}
let ids = context.get("eventIds") || [];
if (ids.includes(id)) return null; // descarta QoS/reenvio duplicado
ids.push(id); if (ids.length > 1000) ids = ids.slice(-1000);
context.set("eventIds", ids);
const estados = context.get("estados") || {};
const chave = equipamento.trim();
const e = estados[chave] || {status:"NORMAL", altas:0, alerta_id:null, severidade:null};
let evento = null;
if (e.status === "NORMAL") {
    e.altas = temp >= 75 ? e.altas + 1 : 0;
    if (e.altas >= 3) {
        e.status = "ABERTO"; e.alerta_id = `ALT-${chave}-${id}`;
        e.severidade = temp >= 90 ? "CRÍTICA" : "ALTA";
        evento = {tipo:"ALERTA_ABERTO", alerta_id:e.alerta_id, severidade:e.severidade};
    }
} else if (temp <= 70) {
    evento = {tipo:"ALERTA_NORMALIZADO", alerta_id:e.alerta_id};
    Object.assign(e, {status:"NORMAL", altas:0, alerta_id:null, severidade:null});
} else if (temp >= 90 && e.severidade !== "CRÍTICA") {
    e.severidade = "CRÍTICA";
    evento = {tipo:"SEVERIDADE_ESCALADA", alerta_id:e.alerta_id, severidade:e.severidade};
}
estados[chave] = e; context.set("estados", estados);
msg.telemetria = {event_id:id, equipment_id:chave, timestamp:p.timestamp,
                  temperature_c:temp, status_alerta:e.status};
if (evento) {
    msg.payload = {...evento, equipment_id:chave, event_id:id,
                   timestamp:p.timestamp || new Date().toISOString(), temperature_c:temp};
    return [msg, null];
}
msg.payload = msg.telemetria;
return [null, msg];


### Exercício no editor Node-RED

1. Monte o fluxo com dois Injects, uma Function de validação e dois Debugs, conforme as seções anteriores.
2. Cole a função de validação no Function configurado com duas saídas. Cole o código acima e configure 2 saídas.
3. Ligue cada saída a um Debug; adicione Catch para erros de validação.
4. Ajuste o Inject para enviar `event_id`, `equipment_id`, `timestamp` e temperatura. Injete 76, 77 e 78 com IDs distintos; reenvie um ID e confirme que não conta novamente.
5. Injete 74, 72 e 69; apenas 69 normaliza. Depois envie três novas altas para demonstrar recorrência.

O MQTT In fornecido usa QoS 0; o Inject liga diretamente ao Switch. Para explorar QoS 1, publique via broker com publicador configurado para QoS 1. A disponibilidade `birthTopic`/`willTopic` indica estado da conexão Node-RED, não o estado de alerta da máquina.


## Atividade e gabarito resumido

Com abertura ≥75 °C, três altas e normalização ≤70 °C, análise `74, 76, 77, 78, 76, 74, 71, 69, 77, 78, 79`. Registre estado, contador e eventos. Acrescente crítico em 90 °C sem remover histerese.

**Gabarito:** 74 mantém NORMAL e zera contador; 76/77 deixam 1/2 altas; 78 abre; 76/74/71 mantém aberto; 69 normaliza; 77/78 iniciam nova persistência; 79 abre nova ocorrência. Duplicata de ID é descartada antes de contar.

## Entrega e referências

Demonstre validação, roteamento, abertura persistente, supressão, reconhecimento sem normalização, escalada, normalização, recorrência e deduplicação. Entregue sequência de entradas/saídas e fluxo atualizado. Contexto local é volátil; eventos fora de ordem, concorrência e persistência durável exigem desenho adicional e validação dos limites com a equipe da planta.

- OASIS, MQTT 3.1.1: <https://docs.oasis-open.org/mqtt/mqtt/v3.1.1/os/mqtt-v3.1.1-os.html>
- Node-RED, mensagens: <https://nodered.org/docs/user-guide/messages>
- Node-RED, Function: <https://nodered.org/docs/user-guide/writing-functions>
- NIST SP 800-82 Rev. 3: <https://csrc.nist.gov/pubs/sp/800/82/r3/final>


# 3. Regras de alerta no Node-RED sem fórmulas

Vamos transformar telemetria em um alerta compreensível no editor visual. A ideia é montar a rota com nós de validação e decisão, reservando o Function para manter a contagem e o ciclo de vida quando o estado exigir memória.

```text
Inject/MQTT --> JSON --> Validar contrato --> regra de alerta
                         |                     |-- NORMAL e temp >= 75: contar
                         |                     |-- ABERTO e temp <= 70: normalizar
                         +--> Catch (erros)    +-- demais: manter estado
                                                   |
                                             Dashboard + Debug
```

Uma Function não é necessária para cada comparação. O Switch pode comparar propriedades simples (por exemplo `msg.payload.temperature_c` maior/igual a 75). Para contagem consecutiva, identificador do alerta e transição de estado, use uma Function ou um subflow que encapsule essa lógica. Evite concentrar validação, roteamento, apresentação e persistência em um único bloco opaco.


## 4. Capturar erros com Catch

No Node-RED, configure a validação para registrar erros com `node.error("mensagem", msg)`. Adicione um no **Catch** no mesmo fluxo e encaminhe sua saída para um Debug de erro e, quando apropriado, para armazenamento/registro operacional. O Catch recebe contexto do erro e a mensagem original; ele não corrige o dado automaticamente.

```text
Validar telemetria -- valido --> regra de alerta
         |
         +-- node.error(msg) --> Catch --> Debug/registro de rejeitados
```

Não ligue erros de validação de volta para a regra, pois isso cria um ciclo e pode transformar uma falha de sensor em uma leitura aceita. Inclua equipamento, event_id (quando disponível), motivo e instante no registro. Evite registrar segredos ou dados desnecessários.


In [ ]:
// Trecho de validação para Function Node-RED; ligue Catch ao erro.
const p = msg.payload;
if (!p || typeof p !== "object" || Array.isArray(p)) {
    node.error("payload deve ser objeto JSON", msg);
    return null;
}
const campos = ["event_id", "equipment_id", "timestamp", "temperature_c"];
const ausentes = campos.filter(campo => p[campo] === undefined || p[campo] === null);
if (ausentes.length) {
    node.error(`campos obrigatórios ausentes: ${ausentes.join(", ")}`, msg);
    return null;
}
if (typeof p.temperature_c !== "number" || !Number.isFinite(p.temperature_c)) {
    node.error("temperature_c deve ser numero finito", msg);
    return null;
}
if (Number.isNaN(Date.parse(p.timestamp)) || !/(Z|[+-]\d\d:\d\d)$/.test(p.timestamp)) {
    node.error("timestamp inválido ou sem fuso", msg);
    return null;
}
msg.validado = true;
return msg;


## 5. Histerese e persistência

A regra da aula abre com três amostras consecutivas de pelo menos 75 C e só normaliza a 70 C ou menos. De 70 a 75 C, conserva o estado atual: essa banda evita que pequenas oscilações abram e fechem alarmes continuamente.

A contagem e por equipamento. Antes da abertura, qualquer amostra abaixo de 75 C reinicia a contagem. Depois de aberto, leituras na banda não fecham o alerta. Abaixo está a simulação de referência; compare as transições com as mensagens do fluxo.


In [ ]:
limite_abertura = 75.0
limite_normalização = 70.0
leituras_para_abrir = 3
estado = {"status": "NORMAL", "altas": 0, "alerta_id": None}

for i, temp in enumerate([76.1, 77.0, 78.4, 74.9, 72.0, 69.5], start=1):
    if estado["status"] == "NORMAL":
        estado["altas"] = estado["altas"] + 1 if temp >= limite_abertura else 0
        if estado["altas"] >= leituras_para_abrir:
            estado.update(status="ABERTO", alerta_id=f"ALT-prensa-01-{i:03d}")
            print(i, temp, "ABRIR", estado["alerta_id"])
        else:
            print(i, temp, "NORMAL / persistência", estado["altas"])
    elif temp <= limite_normalização:
        print(i, temp, "NORMALIZAR", estado["alerta_id"])
        estado.update(status="NORMAL", altas=0, alerta_id=None)
    else:
        print(i, temp, estado["status"], "manter alerta")

estado


A terceira alta abre o alerta. Leituras entre 70 C e 75 C não o fecham; apenas a queda a 70 C ou menos normaliza. Na configuração visual, um Switch pode testar o limiar, mas contador consecutivo e memória de estado precisam de contexto por equipamento; use Function ou subflow para a máquina de estados. O contexto em memória local pode se perder quando o runtime reinicia.

## 6. Reconhecimento, severidade e supressão

- **Aberto:** a condição técnica está ativa.
- **Reconhecido:** operador confirmou ciência; o problema ainda pode existir.
- **Normalizado:** a condição deixou de existir segundo a regra técnica.

Guarde `alerta_id`, instante de abertura, severidade e dados de reconhecimento (usuário e instante). Suprima notificações repetidas para o mesmo alerta aberto. Uma escalada de severidade deve gerar evento mesmo sem nova abertura. Depois de normalizar, uma nova ocorrência deve receber novo identificador.


In [ ]:
// Exemplo de reconhecimento separado da condicao tecnica.
if (msg.ação === "reconhecer") {
    const estados = context.get("estados") || {};
    const e = estados[msg.equipment_id];
    if (!e || e.status !== "ABERTO") {
        node.error("nao ha alerta ativo para reconhecer", msg);
        return null;
    }
    if (!e.reconhecido_por) {
        e.reconhecido_por = msg.usuário;
        e.reconhecido_em = new Date().toISOString();
        estados[msg.equipment_id] = e;
        context.set("estados", estados);
    }
    msg.payload = {tipo:"ALERTA_RECONHECIDO", alerta_id:e.alerta_id,
                   equipment_id:msg.equipment_id, reconhecido_por:e.reconhecido_por,
                   reconhecido_em:e.reconhecido_em};
}
return msg;


## 7. Frontend no Node-RED Dashboard

Para visualizar o ciclo de vida, instale/ative o pacote **Dashboard 2.0** (`@flowfuse/node-red-dashboard`) e crie uma página `Alertas`, um grupo `Prensa` e estes componentes:

- **ui-text**: equipamento, temperatura atual, estado e severidade;
- **ui-gauge**: temperatura em C, faixa sugerida de 0 a 100;
- **ui-table** ou **ui-template**: lista de alertas com ID, abertura, estado e reconhecimento;
- **ui-button**: ação “Reconhecer”, enviando `msg.ação = "reconhecer"`, `msg.equipment_id` e usuário autenticado;
- **ui-notification** (opcional): avisar apenas em abertura e escalada, não a cada leitura.

Encaminhe mensagens validadas para a tela de telemetria e eventos de transição para a tabela/notificação. O botão de reconhecimento deve passar por validação de permissão e registrar quem acionou; não aceite identidade fornecida livremente pelo navegador em um sistema real.

Exemplo de Function de apresentação antes dos widgets:


In [ ]:
// Monta os campos apresentados pelos widgets (nao substitui a lógica de alerta).
const p = msg.payload || {};
msg.payload = {
    equipamento: p.equipment_id || msg.equipment_id || "desconhecido",
    temperatura: p.temperature_c,
    estado: p.status || msg.status_alerta || "NORMAL",
    severidade: p.severidade || "INFO",
    alerta_id: p.alerta_id || null,
    atualizado_em: p.timestamp || new Date().toISOString()
};
return msg;


### Fluxo de interface e teste final

```text
Telemetria validada --> regra --> ui-text + ui-gauge
                              +--> evento abrir/escalar --> ui-table + notificacao
Dashboard button --> acao reconhecer --> Function --> atualizar tabela
Catch --> Debug de erros / registro de rejeitados
```

Teste com as temperaturas `76.1, 77.0, 78.4, 74.9, 72.0, 69.5`. Verifique contador pendente, abertura no terceiro valor, manutenção do alerta na banda de histerese, normalização a 69.5 C e atualização dos widgets. Em seguida, reconheça um alerta ativo e confirme que ele continua aberto até a regra técnica normalizar.

**Limites do exemplo:** os valores são didáticos e não substituem análise de risco da planta. Para uso operacional, configure armazenamento durável/compartilhado, deduplicação por `event_id`, tratamento de mensagens fora de ordem, controle de acesso no Dashboard e validação com a equipe responsável.


### Exemplo de frontend com `ui-template`

No Dashboard 2.0, adicione um `ui-template` ao grupo `Prensa`, selecione tipo de template **Widget** e cole o HTML abaixo. Conecte a ele a saída de apresentação da seção anterior. O cartão atualiza temperatura, equipamento, estado e severidade sempre que recebe uma mensagem. O botão emite uma solicitação de reconhecimento; a Function de reconhecimento deve validar o usuário e o alerta antes de alterar o contexto.


In [ ]:
<template>
  <v-card class="pa-4">
    <div class="text-h6">Alerta industrial</div>
    <div>Equipamento: {{ msg?.payload?.equipamento || "--" }}</div>
    <div class="text-h4">{{ msg?.payload?.temperatura ?? "--" }} °C</div>
    <div>Estado: {{ msg?.payload?.estado || "NORMAL" }}</div>
    <div>Severidade: {{ msg?.payload?.severidade || "INFO" }}</div>
    <div v-if="msg?.payload?.alerta_id">ID: {{ msg.payload.alerta_id }}</div>
    <v-btn class="mt-3" color="primary" @click="reconhecer">
      Reconhecer alerta
    </v-btn>
  </v-card>
</template>

<script>
export default {
  methods: {
    reconhecer() {
      const equipamento = this.msg?.payload?.equipamento;
      if (!equipamento) return;
      this.send({
        topic: "alerta/reconhecer",
        acao: "reconhecer",
        equipment_id: equipamento,
        payload: { equipment_id: equipamento }
      });
    }
  }
}
</script>
